# cAPTure: auditoría estructural del piloto de grafos

Este notebook ejecuta únicamente la Etapa 1 del piloto: construye snapshots de cinco segundos en memoria y audita tamaño, conectividad, multiplicidad, cambio temporal, mezcla benigno/ataque, shortcuts y recursos. No entrena modelos y no accede a Test1, Test2, `train_pub_exf` ni `train_user_prop`.

La ejecución recomendada es: **SMOKE → revisión manual → FULL_DEV**. Los resultados son descriptivos y no aprueban automáticamente el gate estructural.

## 1. Montar Drive y preparar el entorno

Antes de abrir Colab, subir estos cambios a la rama configurada debajo. Esta sección instala sólo las dependencias CPU existentes; no ejecuta tests.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from datetime import datetime, timezone
from pathlib import Path
import json
import subprocess
import sys

REPOSITORY_URL = "https://github.com/tatipar/temporalgnn-nids.git"
REPOSITORY_BRANCH = "feat/capture-feasibility"
PROJECT_ROOT = Path("/content/temporalgnn-nids")
DRIVE_ROOT = Path("/content/drive/MyDrive/capture_gate0")
LOCAL_WORK_ROOT = Path("/content/capture_graph_structural_work")

if not PROJECT_ROOT.exists():
    subprocess.run(
        ["git", "clone", "--branch", REPOSITORY_BRANCH, "--single-branch",
         REPOSITORY_URL, str(PROJECT_ROOT)],
        check=True,
    )

branch = subprocess.check_output(
    ["git", "branch", "--show-current"], cwd=PROJECT_ROOT, text=True
).strip()
if branch != REPOSITORY_BRANCH:
    raise RuntimeError(f"Expected branch {REPOSITORY_BRANCH}, found {branch}.")

required_files = [
    "code/python/requirements-capture.txt",
    "code/python/utils/capture_graph_structural.py",
    "configs/capture_experiment_v1.yaml",
    "configs/capture_packet_schema_v1.yaml",
    "configs/capture_graph_structural_audit_v1.yaml",
]
missing = [name for name in required_files if not (PROJECT_ROOT / name).is_file()]
if missing:
    raise FileNotFoundError(f"Update the Colab repository copy first: {missing}")

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r",
     str(PROJECT_ROOT / "code/python/requirements-capture.txt")],
    check=True,
)
sys.path.insert(0, str(PROJECT_ROOT / "code/python"))

import pandas as pd
from IPython.display import display, Markdown
from utils.capture_graph_structural import run_capture_graph_structural_audit

print("Repository commit:", subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=PROJECT_ROOT, text=True
).strip())
print("CPU structural-audit environment is ready.")

## 2. Vincular los artefactos FULL_DEV preparados

La auditoría usa los Parquet canónicos ya verificados. Procesa un escenario por vez y usa `/content` como staging efímero. `BATCH_SIZE` es un parámetro de ejecución, no una decisión científica; reducirlo si Colab queda corto de RAM.

In [ ]:
MANIFEST_PATH = PROJECT_ROOT / "configs/capture_experiment_v1.yaml"
PACKET_SCHEMA_PATH = PROJECT_ROOT / "configs/capture_packet_schema_v1.yaml"
STRUCTURAL_CONFIG_PATH = (
    PROJECT_ROOT / "configs/capture_graph_structural_audit_v1.yaml"
)
PREPARED_RUN_DIR = (
    DRIVE_ROOT / "prepared_runs/20260917T235058_827743Z_prepare_full_dev"
)
BATCH_SIZE = 250_000

for required_path in (
    MANIFEST_PATH, PACKET_SCHEMA_PATH, STRUCTURAL_CONFIG_PATH, PREPARED_RUN_DIR
):
    if not required_path.exists():
        raise FileNotFoundError(f"Required input is missing: {required_path}")

print("Prepared input:", PREPARED_RUN_DIR)
print("Local staging:", LOCAL_WORK_ROOT)

## 3. Ejecutar SMOKE

El SMOKE procesa `train_empty_conn` y `train_dollar_char`, uno por cada bloque benigno. Para reanudar una corrida existente, reemplazar `None` por su ID exacto.

In [ ]:
SMOKE_RUN_ID = None  # Ejemplo: "20260928T120000_000000Z_graph_structural_smoke"
if SMOKE_RUN_ID is None:
    SMOKE_RUN_ID = (
        datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
        + "_graph_structural_smoke"
    )
SMOKE_OUTPUT_DIR = DRIVE_ROOT / "graph_pilot_runs" / SMOKE_RUN_ID
print("SMOKE run ID:", SMOKE_RUN_ID)
print("SMOKE output:", SMOKE_OUTPUT_DIR)

SMOKE_RESULT = run_capture_graph_structural_audit(
    manifest_path=MANIFEST_PATH,
    packet_schema_path=PACKET_SCHEMA_PATH,
    structural_config_path=STRUCTURAL_CONFIG_PATH,
    prepared_run_dir=PREPARED_RUN_DIR,
    output_dir=SMOKE_OUTPUT_DIR,
    local_work_root=LOCAL_WORK_ROOT,
    mode="SMOKE",
    batch_size=BATCH_SIZE,
)

## 4. Revisar SMOKE

Revisar conservación de filas, tamaños plausibles, Jaccard, componentes, ventanas mixtas, tiempo y RAM. Abrir también `review.md` y `shortcut_audit.json` en Drive antes de aprobar FULL_DEV.

In [ ]:
smoke_scenarios = pd.read_csv(SMOKE_OUTPUT_DIR / "scenario_summary.csv")
smoke_percentiles = pd.read_csv(SMOKE_OUTPUT_DIR / "percentile_summary.csv")
smoke_resources = json.loads(
    (SMOKE_OUTPUT_DIR / "resource_summary.json").read_text()
)
smoke_construction = json.loads(
    (SMOKE_OUTPUT_DIR / "construction_summary.json").read_text()
)

display(smoke_scenarios)
display(smoke_percentiles.loc[
    smoke_percentiles["metric"].isin([
        "nodes", "edges", "weak_components",
        "largest_weak_component_fraction", "mean_multiplicity",
        "adjacent_node_jaccard", "adjacent_pair_jaccard",
    ])
].set_index(["scenario", "metric", "percentile"]))
print("Row conservation:", smoke_construction["row_conservation_passed"])
print(json.dumps(smoke_resources, indent=2)[:12000])
display(Markdown((SMOKE_OUTPUT_DIR / "review.md").read_text()))

## 5. Aprobación manual para FULL_DEV

Cambiar `APPROVE_FULL_DEV` a `True` únicamente después de revisar el SMOKE. La nota queda visible en la sesión; el gate científico definitivo se documenta en el `review.md` de FULL_DEV.

In [ ]:
APPROVE_FULL_DEV = False
SMOKE_REVIEW_NOTE = ""  # Resumir conservación, plausibilidad y recursos observados.

if not APPROVE_FULL_DEV:
    raise RuntimeError(
        "Review the SMOKE artifacts, then set APPROVE_FULL_DEV=True."
    )
if not SMOKE_REVIEW_NOTE.strip():
    raise ValueError("Record a non-empty SMOKE_REVIEW_NOTE before FULL_DEV.")
print("FULL_DEV approved from this reviewed SMOKE:", SMOKE_RUN_ID)

## 6. Ejecutar FULL_DEV

FULL_DEV procesa exclusivamente los cinco escenarios de desarrollo. Cada escenario terminado es inmutable y verificable; para reanudar, usar el mismo `FULL_DEV_RUN_ID`.

In [ ]:
FULL_DEV_RUN_ID = None  # Pegar aquí el ID existente para reanudar.
if FULL_DEV_RUN_ID is None:
    FULL_DEV_RUN_ID = (
        datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
        + "_graph_structural_full_dev"
    )
FULL_DEV_OUTPUT_DIR = DRIVE_ROOT / "graph_pilot_runs" / FULL_DEV_RUN_ID
print("FULL_DEV run ID:", FULL_DEV_RUN_ID)
print("FULL_DEV output:", FULL_DEV_OUTPUT_DIR)

FULL_DEV_RESULT = run_capture_graph_structural_audit(
    manifest_path=MANIFEST_PATH,
    packet_schema_path=PACKET_SCHEMA_PATH,
    structural_config_path=STRUCTURAL_CONFIG_PATH,
    prepared_run_dir=PREPARED_RUN_DIR,
    output_dir=FULL_DEV_OUTPUT_DIR,
    local_work_root=LOCAL_WORK_ROOT,
    mode="FULL_DEV",
    batch_size=BATCH_SIZE,
)

## 7. Revisar FULL_DEV y completar el gate estructural

Las tablas de step son descriptivas: una ventana con varios steps aparece en varios grupos. No interpretar ventanas como observaciones independientes. La decisión final debe escribirse en la copia de `review.md` sin consultar Test1/Test2.

In [ ]:
scenario_summary = pd.read_csv(FULL_DEV_OUTPUT_DIR / "scenario_summary.csv")
percentile_summary = pd.read_csv(FULL_DEV_OUTPUT_DIR / "percentile_summary.csv")
step_summary = pd.read_csv(FULL_DEV_OUTPUT_DIR / "step_topology_summary.csv")
shortcut_audit = json.loads(
    (FULL_DEV_OUTPUT_DIR / "shortcut_audit.json").read_text()
)
resource_summary = json.loads(
    (FULL_DEV_OUTPUT_DIR / "resource_summary.json").read_text()
)

display(scenario_summary.set_index("scenario"))
display(percentile_summary.loc[
    percentile_summary["metric"].isin([
        "nodes", "edges", "weak_components",
        "largest_weak_component_fraction", "mean_multiplicity",
        "branching_node_fraction", "adjacent_node_jaccard",
        "adjacent_pair_jaccard",
    ])
].set_index(["scenario", "metric", "percentile"]))
display(step_summary.head(200))
print("Fold endpoint/pair memorisation diagnostics:")
print(json.dumps(shortcut_audit["fold_memorisation_rules"], indent=2)[:20000])
print("Resource summary:")
print(json.dumps(resource_summary, indent=2))
display(Markdown((FULL_DEV_OUTPUT_DIR / "review.md").read_text()))
print("Review file:", FULL_DEV_OUTPUT_DIR / "review.md")

## 8. Límite de interpretación

Esta auditoría puede mostrar variación, trivialidad, atajos y factibilidad de recursos. No demuestra que una GNN use la topología. La siguiente etapa sólo se implementa después de revisar estos artefactos y congelar política de memoria, checkpoint interno, hiperparámetros y tamaños mínimos de efecto.